# PLAY WITH SQL & LLM

In [1]:
from langchain_community.utilities import SQLDatabase
import os

# set database path
data_folder = "./data"
ddbb_path = os.path.join(data_folder, "recipes.db")

# get database and prompt table names
db = SQLDatabase.from_uri(f"sqlite:///{ddbb_path}")
print(f"database tables are: \n{db.get_usable_table_names()}")

database tables are: 
['cook_technique', 'dish_order', 'ingredient', 'ingredient_category', 'localization', 'recipe', 'recipe_ingredient', 'recipe_ingredient_category']


# MODEL PULLING

Pull or download the `llama3.2:1b` model from the Ollama library.

In [2]:
# download model to fine-tune
import ollama

ollama.pull("llama3.2:1b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

# COUNT ROWS IN TABLE

In [3]:
from langchain.llms import Ollama
from langchain.chains import create_sql_query_chain
import re

# Function to extract only SQL from response markdown.
def extract_sql_from_response(response):
    pattern = r"```sql\n(SELECT .*?);\n```"  # Captura solo la consulta SQL
    match = re.search(pattern, response, re.DOTALL)
    return match.group(1) if match else None

# Load ollama model
llm = Ollama(model="llama3.2:1b", temperature=0) # always same result for same input

# params
tables = ['cook_technique', 'dish_order', 'ingredient', 'ingredient_category', 'localization', 'recipe']
# Iterate in tables and obtain corresponding result
for table in tables:
    chain = create_sql_query_chain(llm, db)
    response = chain.invoke({"question": f"How many {table}s are in {table} table? give sql"})
    sql_query = extract_sql_from_response(response)
    result =db.run(sql_query)
    print(f"---\nIn '{table}' table are : {result} {table}s.")

---
In 'cook_technique' table are : [(15,)] cook_techniques.
---
In 'dish_order' table are : [(3,)] dish_orders.
---
In 'ingredient' table are : [(132,)] ingredients.
---
In 'ingredient_category' table are : [(13,)] ingredient_categorys.
---
In 'localization' table are : [(8,)] localizations.
---
In 'recipe' table are : [(397,)] recipes.


# LIST AND LIMIT ROWS IN TABLE

In [4]:
# params
tables = ['cook_technique', 'dish_order', 'ingredient', 'ingredient_category', 'localization', 'recipe']
number = 10

# Iterate in tables and obtain corresponding result
for table in tables:
    chain = create_sql_query_chain(llm, db)
    response =  chain.invoke({
        "question": (
            f"Give a complete SQL query to get all the records from the single '{table}' table, limiting the result to {number} rows. "
            f"The query should be in the form 'SELECT * FROM {table} LIMIT {number};'"
        )
        })
    sql_query = extract_sql_from_response(response)
    result =db.run(sql_query)
    print(f"---\nIn '{table}' table are:\n {result} {table}s.")

---
In 'cook_technique' table are:
 [(1, 'betea'), (2, 'carpaccio'), (3, 'egosi'), (4, 'entsalada'), (5, 'erre'), (6, 'erregosi'), (7, 'frijitu'), (8, 'hotza'), (9, 'konfitatu'), (10, 'kremak')] cook_techniques.
---
In 'dish_order' table are:
 [(1, 'azkenburukoak'), (2, 'bigarren platerak'), (3, 'lehen platerak')] dish_orders.
---
In 'ingredient' table are:
 [(1, 'antxoa', 1), (2, 'atun-mendrezka', 1), (3, 'atungorri', 1), (4, 'bakailao', 1), (5, 'bakailao-kokotx', 1), (6, 'bakailao-masail', 1), (7, 'barbarin', 1), (8, 'berdela', 1), (9, 'bisigua', 1), (10, 'hegalaburra', 1)] ingredients.
---
In 'ingredient_category' table are:
 [(1, 'arraina'), (2, 'arrautza'), (3, 'arroza'), (4, 'barazkia'), (5, 'barraskiloak'), (6, 'esnekiak'), (7, 'fruta'), (8, 'haragia'), (9, 'itsaskia'), (10, 'lekaleak')] ingredient_categorys.
---
In 'localization' table are:
 [(1, 'afrika'), (2, 'asia'), (3, 'bertakoa'), (4, 'espainia'), (5, 'europa'), (6, 'frantzia'), (7, 'italia'), (8, 'mexiko')] localizations

# LIST SPECIFIED COLUMNS FROM LIMITED TABLE

Select only `recipe` and `url` from `recipes` table.

In [5]:
# params
table = "recipe"
columns = ["recipe", "url"]
number = 3

# create sql chain
chain = create_sql_query_chain(llm, db)

# Create the SQL query instruction
response = chain.invoke({
   "question": (
        f"Give a complete SQL query to get {', '.join(columns)} columns from the single '{table}' table, limiting the result to {number} rows. "
        f"The query should be in the form 'SELECT {', '.join(columns)} FROM {table} LIMIT {number};'"
    )
})

# Extract the SQL query from the response
sql_query = extract_sql_from_response(response)

# Run the SQL query on the database
result = db.run(sql_query)

# Output the result
print(f"---\nIn '{table}' table are:\n{result} {table}s.")

---
In 'recipe' table are:
[('Abakando errea', 'https://eu.wikibooks.org/wiki/Sukaldaritza_liburua/Errezetak/Abakando_errea'), ('Ahuntz-gazta entsalada', 'https://eu.wikibooks.org/wiki/Sukaldaritza_liburua/Errezetak/Ahuntz_gazta_entsalada'), ('Alberjinia beteak haragi-xehatuarekin', 'https://eu.wikibooks.org/wiki/Sukaldaritza_liburua/Errezetak/Alberjinia_beteak_haragi_xehatuarekin')] recipes.


# GROUP DATA BY JOINING

How many `recipes` per each `dish_order`, `cook_technique` and `localization`?

In [27]:
# params
table = "recipe"
group_tables = ["dish_order", "cook_technique", "localization"]

# create sql chain
chain = create_sql_query_chain(llm, db)

# iterate group table. Simple query without join.
for group_table in group_tables:
    # Create the SQL query instruction
    response = chain.invoke({
        "question": (
                    f"Give a complete SQL query to get how many {table}s are in {table} per each {group_table} element."
                    f"The query should use a LEFT JOIN to the {group_table} and return the corresponding {group_table} value instead of just the ID. "
                    f"The query should handle cases where the {group_table}_id is NULL, ensuring those rows are still included in the result. "
                    f"The query should be in the form 'SELECT COALESCE({group_table}.{group_table}, 'Unknown') as name, COUNT(*) FROM {table} "
                    f"LEFT JOIN {group_table} ON {table}.{group_table}_id = {group_table}.{group_table}_id GROUP BY name;'"
                    )
    })
    # Extract the SQL query from the response
    sql_query = extract_sql_from_response(response)
    #print(sql_query)
    
    # Run the SQL query on the database
    result = db.run(sql_query)
    
    # Output the result
    print(f"---\nIn '{table}' table per '{group_table}' element are:\n{result} recipes")

---
In 'recipe' table per 'dish_order' element are:
[('azkenburukoak', 65), ('bigarren platerak', 136), ('lehen platerak', 196)] recipes
---
In 'recipe' table per 'cook_technique' element are:
[('Unknown', 145), ('betea', 23), ('carpaccio', 5), ('egosi', 29), ('entsalada', 38), ('erre', 3), ('erregosi', 3), ('frijitu', 32), ('hotza', 16), ('konfitatu', 2), ('kremak', 32), ('labekatu', 33), ('marian', 3), ('plantxan', 21), ('sueztitu', 4), ('zopak', 8)] recipes
---
In 'recipe' table per 'localization' element are:
[('Unknown', 211), ('afrika', 1), ('asia', 16), ('bertakoa', 99), ('espainia', 25), ('europa', 5), ('frantzia', 17), ('italia', 18), ('mexiko', 5)] recipes


In [ ]:
# params
table = "recipe_ingredient_category"

# obtain joining tables
pos = table.find('_')
first_table = table[0:pos]
second_table = table[pos+1:len(table)]
print("First table:" , first_table)
print("Second table:" , second_table)          

# create sql chain
chain = create_sql_query_chain(llm, db)

# Create the SQL query instruction
response = chain.invoke({
    "question": (
        f"Give a complete SQL query to get how many {first_table}s are in {first_table} table per each {second_table}. "
        f"The query should be in the form 'SELECT {second_table}.{second_table}, COUNT(*) FROM {table} "
        f"JOIN {first_table} ON {first_table}.{first_table}_id = {table}.{first_table}_id "
        f"JOIN {second_table} ON {second_table}.{second_table}_id = {table}.category_id "
        f"GROUP BY {second_table}.{second_table};'"
    )
})

# Extract the SQL query from the response
sql_query = extract_sql_from_response(response)
#print(sql_query)

# Run the SQL query on the database
result = db.run(sql_query)

# Output the result
print(f"---\nIn {tables[0]}s are in {tables[0]} table per each {tables[1]} are:\n{result} recipes")

First table: recipe
Second table: ingredient_category
